# Parent Atlas - `.okf` domain learning challenger, v2 (GRU vs tuned lexical baseline)

**PROPOSAL_ONLY.** Run in Google Colab or a separate Python environment. It does not crawl websites, call an LLM, modify the repo, or write to PostgreSQL/Qdrant/Valkey. It consumes **reviewed** `feature_label`/task examples and the existing checksummed NES domain palette. The bundled synthetic fixture is a smoke test only.

**What changed from v1 (review fixes):**
1. Synthetic labels no longer contain the domain name (v1's `"<DOMAIN> demo"` leaked the answer).
2. Three-way split by independent source group: **train / validation / test**. Model selection (epoch, regularization) uses validation only; the **test set is scored once** at the end.
3. The lexical baseline is tuned on the same validation set as the GRU, plus a majority-class reference.
4. A data-size gate (>=200 rows and >=30 per class, matching the repo floor): below it the status is `INSUFFICIENT_DATA`, never a held-out result.
5. Longer training with patience, so a GRU result is not just an undertrained model.
6. The receipt records library versions, palette revision, per-class reports, split and class counts, chosen hyperparameters, and `calibration: NOT_PERFORMED`.

**Inputs (optional):** `atlas_reviewed_training_examples.jsonl` (fields: `id`, `text`, `feature_label`, `domain`, `source_ref`, `source_group`, `source_revision`, `workspace_revision`, `label_status` = `REVIEWED`; each `source_group` belongs to exactly one domain and groups must be genuinely independent) and the exact generated `nes-domain-palette-v1.json` from `scripts/atlas/data/`. The `.okf` keyword corpus is reference vocabulary only; **do not** auto-label training examples from it.

Do **not** upload private legal evidence, passwords, credentials or proprietary code without reviewing Colab's external processing implications. Do not derive `domain` labels from `feature_label` by a rule: the model would just learn that rule.

In [ ]:
from pathlib import Path
import os, sys, platform, json, re, random, hashlib, copy, math, datetime
from collections import Counter, defaultdict
import numpy as np
import torch
import sklearn
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 20261107
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'sklearn': sklearn.__version__, 'numpy': np.__version__}
print('Versions:', VERSIONS, '| Device:', DEVICE)
# Detect Colab explicitly. Checking Path('/content').exists() is wrong on machines that merely have a /content (or C:\content) folder.
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules
BASE = Path('/content') if IN_COLAB else Path('.')
OUTPUT = BASE / 'atlas_okf_shadow_outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
MIN_ROWS, MIN_PER_CLASS = 200, 30

## 1. Load reviewed examples (or the synthetic fixture)

With no reviewed input, the notebook builds a synthetic fixture whose `feature_label` carries **no domain information** (a hash of the id). All its metrics are non-promotable.

In [ ]:
# Optional Colab interactive upload; uncomment only when needed.
# from google.colab import files
# files.upload()

TRAIN_PATH = BASE / 'atlas_reviewed_training_examples.jsonl'
DEMO_PATH = BASE / 'example-labeled-tasks.jsonl'
if not TRAIN_PATH.exists() and not DEMO_PATH.exists():
    descriptions = {
      'auth': ['check session permissions', 'verify case ownership', 'reject unauthorized login', 'guard user tokens', 'test access control', 'validate auth middleware'],
      'ui': ['render svelte rune', 'check button layout', 'fix keyboard focus', 'test component props', 'inspect modal styling', 'reconcile responsive toolbar'],
      'retrieval': ['rank query chunks', 'compare nearest neighbors', 'evaluate semantic search', 'test recall at ten', 'merge RRF candidates', 'trace hybrid search'],
      'network': ['check http timeout', 'retry grpc request', 'validate rpc payload', 'trace web socket', 'inspect api endpoint', 'verify tls session'],
      'database': ['review postgres join', 'test drizzle schema', 'verify transaction rollback', 'inspect pgvector index', 'reconcile sql migration', 'check database identity'],
      'cache': ['invalidate valkey key', 'measure cache hit', 'bound prefix warming', 'test expiry fallback', 'inspect LRU budget', 'verify centroid revision'],
      'agent': ['route mcp tool', 'compose ACE evidence', 'approve repair proposal', 'build tool selector', 'check agent execution', 'validate action receipt'],
      'graph': ['expand graph edges', 'calculate pagerank', 'analyze dependency graph', 'audit node provenance', 'trace concept joins', 'verify graph traversal'],
      'ml': ['test torch tensor', 'benchmark cuda gemm', 'train domain classifier', 'validate embedding model', 'measure quantization error', 'score cross encoder'],
      'general': ['update documentation note', 'review project overview', 'check broad guidance', 'write release summary', 'organize work notes', 'prepare team briefing'],
    }
    synthetic = []
    for domain, texts in descriptions.items():
        for i, t in enumerate(texts):
            rid = f'demo:{domain}:{i}'
            # Neutral label: a hash of the id, so it carries NO domain information.
            synthetic.append({'id': rid, 'text': t, 'feature_label': 'Demo feature ' + hashlib.md5(rid.encode()).hexdigest()[:6],
              'domain': domain, 'source_ref': f'demo/{domain}/{i}.md', 'source_group': f'demo:{domain}:{i}',
              'source_revision': 'SYNTHETIC', 'workspace_revision': 'SYNTHETIC', 'label_status': 'EXAMPLE_ONLY'})
    DEMO_PATH.write_text(''.join(json.dumps(x) + '\n' for x in synthetic), encoding='utf-8')

using_demo = not TRAIN_PATH.exists()
path = DEMO_PATH if using_demo else TRAIN_PATH
rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
required = {'id','text','feature_label','domain','source_ref','source_group','source_revision','workspace_revision','label_status'}
assert rows, 'No rows'
assert all(required.issubset(r) for r in rows), 'Missing required fields'
assert len(set(r['id'] for r in rows)) == len(rows), 'Duplicate IDs'
expected_status = 'EXAMPLE_ONLY' if using_demo else 'REVIEWED'
assert all(r['label_status'] == expected_status for r in rows), f'every row must have label_status={expected_status}'
group_domains = defaultdict(set)
for r in rows: group_domains[r['source_group']].add(r['domain'])
assert all(len(d) == 1 for d in group_domains.values()), 'A source_group must belong to exactly one domain'

class_counts = Counter(r['domain'] for r in rows)
if using_demo: DATA_STATUS = 'DEMO_ONLY'
elif len(rows) < MIN_ROWS or min(class_counts.values()) < MIN_PER_CLASS: DATA_STATUS = 'INSUFFICIENT_DATA'
else: DATA_STATUS = 'HELD_OUT_PROPOSAL_ONLY'
print('Rows:', len(rows), '| mode:', 'SYNTHETIC_DEMO' if using_demo else 'REVIEWED_INPUT', '| data status:', DATA_STATUS)
print('Per-class counts:', dict(sorted(class_counts.items())))

## 2. Verify palette authority

`lutByte` (domain routing) and `nesHex` (display colour) are separate fields; `0x00` means unresolved. The notebook refuses unknown domain names and never trains new palette bytes. If the exact palette file is absent it still trains, but emits no byte/colour mapping.

In [ ]:
PALETTE_PATH = BASE / 'nes-domain-palette-v1.json'
PALETTE = None
if PALETTE_PATH.is_file():
    PALETTE = json.loads(PALETTE_PATH.read_text(encoding='utf-8'))
    expect = PALETTE.get('checksum', '')
    body = {k: v for k, v in PALETTE.items() if k != 'checksum'}
    actual = 'sha256:' + hashlib.sha256(json.dumps(body, separators=(',', ':'), ensure_ascii=False).encode()).hexdigest()
    assert actual == expect, f'NES palette checksum failed: {actual}'
    assert PALETTE['schema'] == 'atlas.nes-domain-palette.v1'
    assert PALETTE['unresolved']['lutByte'] == 0
    print('Palette:', PALETTE['revision'], 'validated:', actual)
else:
    print('NES palette not supplied - no byte/colour mapping will be emitted')

allowed_domains = set(PALETTE['domains']) if PALETTE is not None else {'auth','ui','retrieval','network','database','cache','agent','graph','ml','general'}
assert all(r['domain'] in allowed_domains for r in rows), 'Unrecognized domain labels'

## 3. Leakage-aware three-way split and the tuned lexical baseline

Split **by source group** within each domain into train / validation / test (>=5 groups per domain required). Epoch choice and regularization are selected on validation only. The test split is scored **once** at the end. The lexical classifier is the baseline to beat; a GRU is not automatically better.

In [ ]:
by_domain = defaultdict(lambda: defaultdict(list))
for r in rows: by_domain[r['domain']][r['source_group']].append(r)
assert all(len(g) >= 5 for g in by_domain.values()), 'Need >=5 independent source groups per domain (train/validation/test)'
train_rows, val_rows, test_rows = [], [], []
for domain, groups in sorted(by_domain.items()):
    names = sorted(groups)
    random.Random(SEED + sum(map(ord, domain))).shuffle(names)
    n_test = max(1, round(len(names) * .2)); n_val = max(1, round(len(names) * .2))
    test_names = set(names[:n_test]); val_names = set(names[n_test:n_test + n_val])
    for name, rs in groups.items():
        (test_rows if name in test_names else val_rows if name in val_names else train_rows).extend(rs)
groups_of = lambda rs: {r['source_group'] for r in rs}
assert groups_of(train_rows).isdisjoint(groups_of(val_rows)) and groups_of(train_rows).isdisjoint(groups_of(test_rows)) and groups_of(val_rows).isdisjoint(groups_of(test_rows))
labels = sorted(by_domain)
label_to_i = {name: i for i, name in enumerate(labels)}
text_of = lambda r: f"{r['feature_label']} | {r['text']}"  # no source ids as model input
Xtr = [text_of(r) for r in train_rows]; Xva = [text_of(r) for r in val_rows]; Xte = [text_of(r) for r in test_rows]
ytr = [label_to_i[r['domain']] for r in train_rows]; yva = [label_to_i[r['domain']] for r in val_rows]; yte = [label_to_i[r['domain']] for r in test_rows]

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=12000)
Xtrain = vectorizer.fit_transform(Xtr); Xvalid = vectorizer.transform(Xva); Xtest = vectorizer.transform(Xte)
best = None
for C in [0.1, 1.0, 10.0, 100.0]:
    clf = LogisticRegression(C=C, max_iter=1000, random_state=SEED).fit(Xtrain, ytr)
    f = f1_score(yva, clf.predict(Xvalid), average='macro', zero_division=0)
    if best is None or f > best[0] + 1e-9: best = (f, C, clf)
base_val_f1, base_C, baseline = best
majority = DummyClassifier(strategy='most_frequent').fit(Xtrain, ytr)
print('Split rows -> train', len(train_rows), '| validation', len(val_rows), '| test', len(test_rows), '| labels', labels)
print('Baseline selected on validation: C =', base_C, 'validation macro-F1 =', round(base_val_f1, 4))

## 4. Byte-level PyTorch GRU challenger

Every UTF-8 byte is a token (1-256); 0 is padding; truncate to 256 bytes (truncation is counted). Training runs up to 80 epochs with patience 15; the best epoch is chosen on **validation** macro-F1. This is a classification model, not an EmbeddingGemma substitute and not a TurboVec codec. With only a few dozen rows a GRU is not expected to learn; treat a poor result as "insufficient data", not as evidence against GRUs.

In [ ]:
MAX_BYTES = 256
def encode_texts(texts):
    encoded = []; lengths = []; truncated = 0
    for t in texts:
        bs = t.encode('utf-8'); truncated += int(len(bs) > MAX_BYTES)
        tokens = [b + 1 for b in bs[:MAX_BYTES]]
        lengths.append(max(1, len(tokens))); encoded.append((tokens + [0] * MAX_BYTES)[:MAX_BYTES])
    return torch.tensor(encoded, dtype=torch.long), torch.tensor(lengths, dtype=torch.long), truncated

x_train, len_train, trunc_train = encode_texts(Xtr)
x_val, len_val, trunc_val = encode_texts(Xva)
x_test, len_test, trunc_test = encode_texts(Xte)
print('Truncation:', trunc_train, 'train,', trunc_val, 'validation,', trunc_test, 'test')

class ByteGRU(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.embed = nn.Embedding(257, 32, padding_idx=0)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.head = nn.Linear(64, num_classes)
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.embed(x), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        return self.head(h[-1])

MAX_EPOCHS, PATIENCE, LR = 80, 15, 0.003
model = ByteGRU(len(labels)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()
dataloader = DataLoader(TensorDataset(x_train, len_train, torch.tensor(ytr)), batch_size=16, shuffle=True)
best_f1 = -1.; best_state = None; best_epoch = None; bad = 0
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    for xb, lb, yb in dataloader:
        xb, lb, yb = xb.to(DEVICE), lb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss_fn(model(xb, lb), yb).backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    model.eval()
    with torch.inference_mode():
        preds = model(x_val.to(DEVICE), len_val.to(DEVICE)).argmax(-1).cpu().tolist()
    f1 = f1_score(yva, preds, average='macro', zero_division=0)
    if f1 > best_f1 + 1e-6: best_f1 = f1; best_epoch = epoch; best_state = copy.deepcopy(model.state_dict()); bad = 0
    else: bad += 1
    if epoch % 10 == 0 or bad == 0: print(f'epoch={epoch:02d} validation_macro_f1={f1:.4f}')
    if bad >= PATIENCE: break
model.load_state_dict(best_state)
print('GRU selected epoch', best_epoch, '| validation macro-F1', round(best_f1, 4), '| baseline validation macro-F1', round(base_val_f1, 4))

## 5. Score the held-out test split once; export **proposal-only** results

The confidence threshold is a configurable proposal criterion, **not** calibration. The model cannot create a `feature_label`, `task_id`, `lutByte` or `nesHex`. Predictions are exported for the **test split only**.

In [ ]:
model.eval()
with torch.inference_mode():
    p = torch.softmax(model(x_test.to(DEVICE), len_test.to(DEVICE)).float(), dim=-1).cpu().numpy()
gru_test_preds = p.argmax(-1).tolist()
base_test_preds = baseline.predict(Xtest); maj_test_preds = majority.predict(Xtest)
macro = lambda y, yp: float(f1_score(y, yp, average='macro', zero_division=0))
report = lambda y, yp: classification_report(y, yp, labels=list(range(len(labels))), target_names=labels, output_dict=True, zero_division=0)
test_scores = {
    'majority_class': {'accuracy': float(accuracy_score(yte, maj_test_preds)), 'macro_f1': macro(yte, maj_test_preds)},
    'tfidf_logistic': {'accuracy': float(accuracy_score(yte, base_test_preds)), 'macro_f1': macro(yte, base_test_preds), 'selected_C': base_C},
    'byte_gru': {'accuracy': float(accuracy_score(yte, gru_test_preds)), 'macro_f1': macro(yte, gru_test_preds), 'selected_epoch': best_epoch},
}
print(json.dumps(test_scores, indent=2))

state_path = OUTPUT / 'byte_gru_proposal_state.pt'
torch.save(model.cpu().state_dict(), state_path)
state_sha = hashlib.sha256(state_path.read_bytes()).hexdigest()
palette_sha = PALETTE.get('checksum') if PALETTE else None
input_sha = hashlib.sha256(path.read_bytes()).hexdigest()
threshold = 0.70
proposals = []
for row, pr in zip(test_rows, p):
    idx = int(np.argmax(pr)); confidence = float(pr[idx]); pred = labels[idx]
    admitted_label = pred if confidence >= threshold else 'UNRESOLVED'
    nes_entry = (PALETTE['domains'].get(pred) if PALETTE is not None and admitted_label != 'UNRESOLVED' else None)
    proposals.append({
      'schema': 'atlas.okf.domain-proposal.v1', 'status': 'PROPOSAL_ONLY', 'split': 'test',
      'id': row['id'], 'feature_label': row['feature_label'], 'source_ref': row['source_ref'],
      'source_revision': row['source_revision'], 'workspace_revision': row['workspace_revision'],
      'predicted_domain': admitted_label, 'raw_predicted_domain': pred, 'confidence_uncalibrated': round(confidence, 6),
      'lutByte': nes_entry['lutByte'] if nes_entry else (0 if PALETTE else None),
      'nesHex': nes_entry['nesHex'] if nes_entry else None,
      'palette_revision': PALETTE['revision'] if PALETTE else None,
      'model_sha256': state_sha, 'canonical_authority': False
    })
metrics = {
    'schema': 'atlas.okf.learning-shadow-receipt.v2',
    'status': DATA_STATUS, 'generated_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'versions': VERSIONS, 'seed': SEED, 'device': str(DEVICE),
    'rows': len(rows), 'class_counts': dict(sorted(class_counts.items())),
    'min_rows_required': MIN_ROWS, 'min_per_class_required': MIN_PER_CLASS,
    'train_count': len(train_rows), 'validation_count': len(val_rows), 'test_count': len(test_rows),
    'source_groups': {'train': len(groups_of(train_rows)), 'validation': len(groups_of(val_rows)), 'test': len(groups_of(test_rows))},
    'splits_disjoint_by_source_group': True,
    'labels': labels, 'input_sha256': input_sha,
    'palette_revision': PALETTE['revision'] if PALETTE else None, 'palette_checksum': palette_sha,
    'hyperparameters': {'gru': {'max_bytes': MAX_BYTES, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'lr': LR, 'selected_epoch': best_epoch}, 'tfidf_logistic': {'selected_C': base_C}},
    'validation_macro_f1': {'tfidf_logistic': float(base_val_f1), 'byte_gru': float(best_f1)},
    'test_scores': test_scores,
    'per_class_test_report': {'tfidf_logistic': report(yte, base_test_preds), 'byte_gru': report(yte, gru_test_preds)},
    'truncation': {'train': trunc_train, 'validation': trunc_val, 'test': trunc_test},
    'calibration': 'NOT_PERFORMED', 'confidence_threshold': threshold,
    'model_sha256': state_sha, 'promotion_allowed': False,
    'pgvector_writes': 0, 'qdrant_writes': 0, 'valkey_writes': 0
}
(OUTPUT / 'learning-shadow-receipt.json').write_text(json.dumps(metrics, indent=2) + '\n')
(OUTPUT / 'domain-proposals.jsonl').write_text(''.join(json.dumps(x) + '\n' for x in proposals))
print('Status:', DATA_STATUS, '| saved:', sorted(f.name for f in OUTPUT.iterdir()))

### Reading the result
- `DEMO_ONLY` and `INSUFFICIENT_DATA` results are plumbing checks. They are not evidence about classifier quality and must not be promoted.
- Even `HELD_OUT_PROPOSAL_ONLY` is a **proposal**: admission needs independent review of the labels, a frozen judged task set, calibration, and comparison with EmbeddingGemma 2 and the deterministic taxonomy lane.
- Keep `lutByte`/`nesHex` authority in `scripts/atlas/data/nes-domain-palette-v1.json`; this notebook only looks them up.